In [1]:
import os
import json
import copy
import random
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

RANDOM_STATE = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATA_PATH = "data/compiled_model_ready/MR_cities_with_country_worldpop_v1.csv"
ARTIFACT_DIR = "artifacts/geo_mlp_safety_worldpop_v1"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

Device: cpu


In [2]:
cities = pd.read_csv(DATA_PATH)
print("Cities with worldpop features:", cities.shape)
print(cities.columns)

cities.head()

Cities with worldpop features: (509, 26)
Index(['city', 'country', 'country_norm', 'lat', 'lon', 'crime_index',
       'safety_index', 'crimeindex_2023', 'crimeindex_2020',
       'safetyindex_2020', 'age_0_14', 'age_15_64', 'age_65_plus',
       'population', 'density_per_km2', 'num_cities_25km', 'sum_pop_25km',
       'pop_gravity_25km', 'num_cities_50km', 'sum_pop_50km',
       'pop_gravity_50km', 'num_cities_100km', 'sum_pop_100km',
       'pop_gravity_100km', 'dist_to_nearest_large_city',
       'pop_of_nearest_large_city'],
      dtype='object')


,city,country,country_norm,lat,lon,crime_index,safety_index,crimeindex_2023,crimeindex_2020,safetyindex_2020,...,sum_pop_25km,pop_gravity_25km,num_cities_50km,sum_pop_50km,pop_gravity_50km,num_cities_100km,sum_pop_100km,pop_gravity_100km,dist_to_nearest_large_city,pop_of_nearest_large_city
0,Caracas,Venezuela,venezuela,10.506093,-66.914601,83.6,16.4,83.76,84.49,15.51,...,5180367.0,3.533040e+05,19,6465431.0,3.541946e+05,34,9243588.0,3.546679e+05,3.079201,3242000.0
1,Pretoria,South Africa,south africa,-25.745928,28.187910,82.0,18.0,76.86,77.49,22.51,...,3087153.0,3.633554e+09,18,4257637.0,3.633554e+09,41,16239880.0,3.633558e+09,0.026849,2818100.0
2,Durban,South Africa,south africa,-29.861825,31.009909,81.0,19.0,76.86,77.49,22.51,...,1157022.0,4.282806e+04,12,1205953.0,4.288017e+04,22,2265236.0,4.311689e+04,4.543538,838634.0
3,Port Moresby,Papua New Guinea,papua new guinea,-9.474330,147.159950,80.7,19.3,80.79,81.93,18.07,...,317374.0,1.985452e+05,1,317374.0,1.985452e+05,1,317374.0,1.985452e+05,2093.301758,2706966.0
4,Johannesburg,South Africa,south africa,-26.205000,28.049722,80.7,19.3,76.86,77.49,22.51,...,11476027.0,4.508328e+07,25,12399809.0,4.508407e+07,42,15894194.0,4.508523e+07,0.416635,7860781.0


In [3]:
# Core geo + country features (same as lower bound)
base_feature_cols = [
    "lat",
    "lon",
    "crimeindex_2020",
    "crimeindex_2023",
    "safetyindex_2020",
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",        # country-level population
    "density_per_km2",
]

# New population-field features from worldcities
worldpop_feature_cols = [
    "num_cities_25km", "sum_pop_25km", "pop_gravity_25km",
    "num_cities_50km", "sum_pop_50km", "pop_gravity_50km",
    "num_cities_100km", "sum_pop_100km", "pop_gravity_100km",
    "dist_to_nearest_large_city", "pop_of_nearest_large_city",
]

feature_cols = base_feature_cols + worldpop_feature_cols
target_col = "safety_index"

# Drop rows with missing features/target
cities_model = cities.dropna(subset=feature_cols + [target_col]).copy()
print("Model rows:", cities_model.shape)

X = cities_model[feature_cols].values.astype(np.float32)
y = cities_model[target_col].values.astype(np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)

Model rows: (509, 26)
X shape: (509, 21)
y shape: (509,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Train: torch.Size([407, 21]) torch.Size([407, 1])
Test: torch.Size([102, 21]) torch.Size([102, 1])


In [5]:
def make_loaders(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, batch_size=32):
    train_ds = TensorDataset(X_train_tensor, y_train_tensor)
    test_ds = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

    return train_loader, test_loader

In [6]:
class GeoMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [7]:
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            out = model(xb)
            preds.append(out.cpu().numpy())
            targets.append(yb.cpu().numpy())
    preds = np.vstack(preds).ravel()
    targets = np.vstack(targets).ravel()
    rmse = np.sqrt(mean_squared_error(targets, preds))
    mae = mean_absolute_error(targets, preds)
    r2 = r2_score(targets, preds)
    return rmse, mae, r2, preds, targets

In [8]:
def train_one_model(
    model_name,
    hidden_dims=(64, 32),
    dropout=0.2,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=32,
    n_epochs=300,
    verbose=True,
):
    seed_everything(RANDOM_STATE)

    train_loader, test_loader = make_loaders(
        X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, batch_size=batch_size
    )

    model = GeoMLP(
        input_dim=X_train_tensor.shape[1],
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_rmse = float("inf")
    best_state = None
    history = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        if epoch % 20 == 0 or epoch == n_epochs:
            train_rmse, train_mae, train_r2, _, _ = evaluate(model, train_loader)
            test_rmse, test_mae, test_r2, _, _ = evaluate(model, test_loader)

            history.append({
                "epoch": epoch,
                "train_rmse": train_rmse,
                "train_mae": train_mae,
                "train_r2": train_r2,
                "test_rmse": test_rmse,
                "test_mae": test_mae,
                "test_r2": test_r2,
            })

            if verbose:
                print(
                    f"{model_name} | Epoch {epoch:03d} | "
                    f"Train RMSE {train_rmse:.2f}, R2 {train_r2:.3f} | "
                    f"Test RMSE {test_rmse:.2f}, R2 {test_r2:.3f}"
                )

            if test_rmse < best_rmse:
                best_rmse = test_rmse
                best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    final_rmse, final_mae, final_r2, preds, targets = evaluate(model, test_loader)

    result = {
        "model_name": model_name,
        "hidden_dims": hidden_dims,
        "dropout": dropout,
        "lr": lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "n_epochs": n_epochs,
        "best_test_rmse": final_rmse,
        "best_test_mae": final_mae,
        "best_test_r2": final_r2,
        "best_state_dict": copy.deepcopy(model.state_dict()),
        "history": history,
        "preds": preds,
        "targets": targets,
    }

    return result

In [9]:
variant_configs = [
    {"model_name": "safety_pop_v1_e200", "hidden_dims": (64, 32), "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32, "n_epochs": 200},
    {"model_name": "safety_pop_v1_e300", "hidden_dims": (64, 32), "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32, "n_epochs": 300},
    {"model_name": "safety_pop_v1_e400", "hidden_dims": (64, 32), "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32, "n_epochs": 400},
    {"model_name": "safety_pop_v1_low_dropout", "hidden_dims": (64, 32), "dropout": 0.1, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32, "n_epochs": 300},
    {"model_name": "safety_pop_v1_higher_wd", "hidden_dims": (64, 32), "dropout": 0.2, "lr": 1e-3, "weight_decay": 5e-4, "batch_size": 32, "n_epochs": 300},
    {"model_name": "safety_pop_v1_small", "hidden_dims": (32, 16), "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32, "n_epochs": 300},
    {"model_name": "safety_pop_v1_wide", "hidden_dims": (128, 64), "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32, "n_epochs": 300},
    {"model_name": "safety_pop_v1_deeper", "hidden_dims": (128, 64, 32), "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32, "n_epochs": 300},
    {"model_name": "safety_pop_v1_narrow_deep", "hidden_dims": (64, 32, 16), "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32, "n_epochs": 300},
    {"model_name": "safety_pop_v1_wide_low_dropout", "hidden_dims": (128, 64), "dropout": 0.1, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32, "n_epochs": 300},
]

In [10]:
all_results = []

for cfg in variant_configs:
    print("\n" + "=" * 80)
    print("Training:", cfg["model_name"])
    print("=" * 80)
    result = train_one_model(**cfg, verbose=True)
    all_results.append(result)


Training: safety_pop_v1_e200
safety_pop_v1_e200 | Epoch 020 | Train RMSE 15.17, R2 0.005 | Test RMSE 16.17, R2 -0.074
safety_pop_v1_e200 | Epoch 040 | Train RMSE 12.06, R2 0.371 | Test RMSE 13.85, R2 0.212
safety_pop_v1_e200 | Epoch 060 | Train RMSE 11.02, R2 0.475 | Test RMSE 12.91, R2 0.315
safety_pop_v1_e200 | Epoch 080 | Train RMSE 10.45, R2 0.528 | Test RMSE 12.39, R2 0.369
safety_pop_v1_e200 | Epoch 100 | Train RMSE 9.97, R2 0.571 | Test RMSE 12.01, R2 0.408
safety_pop_v1_e200 | Epoch 120 | Train RMSE 9.62, R2 0.600 | Test RMSE 11.76, R2 0.432
safety_pop_v1_e200 | Epoch 140 | Train RMSE 9.57, R2 0.604 | Test RMSE 11.63, R2 0.445
safety_pop_v1_e200 | Epoch 160 | Train RMSE 9.41, R2 0.617 | Test RMSE 11.61, R2 0.446
safety_pop_v1_e200 | Epoch 180 | Train RMSE 9.20, R2 0.634 | Test RMSE 11.28, R2 0.477
safety_pop_v1_e200 | Epoch 200 | Train RMSE 9.09, R2 0.643 | Test RMSE 11.15, R2 0.490

Training: safety_pop_v1_e300
safety_pop_v1_e300 | Epoch 020 | Train RMSE 15.17, R2 0.005 | Tes

In [11]:
results_df = pd.DataFrame([
    {
        "model_name": r["model_name"],
        "hidden_dims": str(r["hidden_dims"]),
        "dropout": r["dropout"],
        "lr": r["lr"],
        "weight_decay": r["weight_decay"],
        "batch_size": r["batch_size"],
        "n_epochs": r["n_epochs"],
        "rmse": r["best_test_rmse"],
        "mae": r["best_test_mae"],
        "r2": r["best_test_r2"],
    }
    for r in all_results
])

results_df = results_df.sort_values(by="rmse", ascending=True).reset_index(drop=True)
print(results_df)

# pick best
best_result = min(all_results, key=lambda x: x["best_test_rmse"])

print("\nBest worldpop model:")
print("Name:", best_result["model_name"])
print("RMSE:", round(best_result["best_test_rmse"], 4))
print("MAE:", round(best_result["best_test_mae"], 4))
print("R2:", round(best_result["best_test_r2"], 4))

# save weights, scaler, config, results
weights_path = os.path.join(ARTIFACT_DIR, f"{best_result['model_name']}_best_weights.pt")
torch.save(best_result["best_state_dict"], weights_path)

scaler_path = os.path.join(ARTIFACT_DIR, f"{best_result['model_name']}_scaler.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)

config_to_save = {
    "model_name": best_result["model_name"],
    "feature_cols": feature_cols,
    "target_col": target_col,
    "hidden_dims": list(best_result["hidden_dims"]),
    "dropout": best_result["dropout"],
    "lr": best_result["lr"],
    "weight_decay": best_result["weight_decay"],
    "batch_size": best_result["batch_size"],
    "n_epochs": best_result["n_epochs"],
    "rmse": best_result["best_test_rmse"],
    "mae": best_result["best_test_mae"],
    "r2": best_result["best_test_r2"],
    "data_path": DATA_PATH,
    "random_state": RANDOM_STATE,
    "notes": "Safety Geo-MLP with worldcities population-field features, no city-level crime.",
}

config_path = os.path.join(ARTIFACT_DIR, f"{best_result['model_name']}_config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)

results_path = os.path.join(ARTIFACT_DIR, "geo_mlp_safety_worldpop_variant_results.csv")
results_df.to_csv(results_path, index=False)

print("\nSaved:")
print(weights_path)
print(scaler_path)
print(config_path)
print(results_path)

                       model_name    hidden_dims  dropout     lr  \
0              safety_pop_v1_e400       (64, 32)      0.2  0.001   
1       safety_pop_v1_low_dropout       (64, 32)      0.1  0.001   
2            safety_pop_v1_deeper  (128, 64, 32)      0.2  0.001   
3  safety_pop_v1_wide_low_dropout      (128, 64)      0.1  0.001   
4              safety_pop_v1_e300       (64, 32)      0.2  0.001   
5         safety_pop_v1_higher_wd       (64, 32)      0.2  0.001   
6       safety_pop_v1_narrow_deep   (64, 32, 16)      0.2  0.001   
7              safety_pop_v1_wide      (128, 64)      0.2  0.001   
8              safety_pop_v1_e200       (64, 32)      0.2  0.001   
9             safety_pop_v1_small       (32, 16)      0.2  0.001   

   weight_decay  batch_size  n_epochs       rmse       mae        r2  
0        0.0001          32       400  10.797747  8.495049  0.521106  
1        0.0001          32       300  10.891745  8.572493  0.512732  
2        0.0001          32       300 

In [12]:
with open(config_path, "r") as f:
    saved_config = json.load(f)

with open(scaler_path, "rb") as f:
    loaded_scaler = pickle.load(f)

best_model = GeoMLP(
    input_dim=len(saved_config["feature_cols"]),
    hidden_dims=tuple(saved_config["hidden_dims"]),
    dropout=saved_config["dropout"],
).to(device)

best_model.load_state_dict(torch.load(weights_path, map_location=device))
best_model.eval()

GeoMLP(
  (net): Sequential(
    (0): Linear(in_features=21, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [13]:
def predict_safety_for_city_row(row):
    x_raw = row[feature_cols].values.astype(np.float32).reshape(1, -1)
    x_scaled = loaded_scaler.transform(x_raw)
    x_tensor = torch.tensor(x_scaled, dtype=torch.float32).to(device)
    with torch.no_grad():
        pred = best_model(x_tensor).cpu().numpy().ravel()[0]
    return float(pred)

sample_row = cities_model.sample(1, random_state=RANDOM_STATE).iloc[0]
print(sample_row["city"], sample_row["country"])
print("True safety:", sample_row[target_col])
print("Pred safety:", predict_safety_for_city_row(sample_row))

Liege Belgium
True safety: 39.6
Pred safety: 54.14488983154297


# How this model differs from lower & upper bounds

### Upper bound

Features: geo + country + city crime_index (same row).

Target: safety_index.

Effect: almost perfect R² (~0.995) because safety ≈ f(crime) at city level.

Problem: not deployable for arbitrary lat/lon (you don’t know that city’s true crime index at inference).

### Lower bound

Features: geo + country only (lat, lon, national crime/safety, age splits, country pop/density).

Target: safety_index.

Effect: R² ≈ 0.53, RMSE ≈ 10.6 in your runs.

Interpretation: “What can we do with country‑level context + coordinates alone?”

### Worldpop v3 (this model)

Features: lower‑bound features plus dense population-field features derived from all 44k world cities:

counts/sums of nearby cities at multiple radii,

gravity‑style population exposure,

distance and size of nearest large city.

Still no city‑level crime as a direct feature.

Effect you should see: RMSE lower and R² higher than the lower bound, because the model now knows whether a point is in a dense metro corridor, isolated town, near a megacity, etc., which literature shows is highly predictive for crime/safety patterns

## Big picture takeaways
All worldpop variants converge to very similar performance: test RMSE in the ~10.8–11.2 band, R² around 0.50–0.52.

Almost identical to lower‑bound Geo‑MLP (~10.54 RMSE, R² ≈ 0.54), meaning the current population field features did not give a clear boost yet

Training RMSE improves steadily with more epochs and wider/deeper nets, but test RMSE/R² plateau in the same range, so you’re limited by signal/noise, not capacity.

So the worldpop model is working and stable, but it hasn’t unlocked extra predictive power yet compared to your baseline.

## Next steps

Try richer spatial signals, not more epochs, layers, ect...

KNN derived crime/safety features from ~500 labeled cities (nearest‑neighbor field).

Smarter use of worldpop (e.g., per‑country normalization, log‑population, or fewer, better‑targeted radii).